# 02 — Training & Evaluation

**Project:** Post-Only (A) vs Trajectory (B) Supervision for Continual Tool-Use Learning

This notebook trains on 6 sequential domain blocks and evaluates after each.

In [7]:
# ============================================================
# CONFIGURATION
# ============================================================
CONDITION = "B"   # "A", "B", or "A+"
SEED = 42

In [8]:
!pip install -q transformers accelerate peft bitsandbytes trl huggingface_hub tqdm

In [9]:
import json
import os
import re
import random
import time
import pickle
import numpy as np
from tqdm.auto import tqdm

import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, DataCollatorForLanguageModeling, Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import Dataset

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Condition: {CONDITION} | Seed: {SEED}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Condition: B | Seed: 42
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


## 1. Load Preprocessed Data

In [10]:
with open('preprocessed_data/preprocessed.pkl', 'rb') as f:
    data = pickle.load(f)

blocks = data['blocks']
config = data['config']
MODEL_NAME = config['model_name']
MAX_SEQ_LEN = config['max_seq_len']
NUM_BLOCKS = config['num_blocks']
BASE_EPOCHS = config['base_epochs']

print(f"Model: {MODEL_NAME}")
print(f"Blocks: {NUM_BLOCKS}, Seq len: {MAX_SEQ_LEN}")
print(f"\nBlock sizes:")
for b in blocks:
    print(f"  D{b['block_id']}: {len(b['train_a'])} train, {len(b['eval_a'])} eval")

Model: mistralai/Mistral-7B-Instruct-v0.3
Blocks: 6, Seq len: 2048

Block sizes:
  D1: 448 train, 113 eval
  D2: 471 train, 118 eval
  D3: 467 train, 117 eval
  D4: 459 train, 115 eval
  D5: 469 train, 118 eval
  D6: 463 train, 116 eval


In [11]:
def get_train_texts(block, condition):
    if condition in ('A', 'A+'):
        return block['train_a'], block['train_a_prompt_lens']
    return block['train_b'], block['train_b_prompt_lens']

def get_eval_texts(block, condition):
    if condition in ('A', 'A+'):
        return block['eval_a'], block['eval_a_prompt_lens']
    return block['eval_b'], block['eval_b_prompt_lens']

def get_epochs(block, condition):
    if condition == 'A+':
        return block['aplus_epochs']
    return BASE_EPOCHS

## 2. Load Model

In [12]:
ATTN_IMPL = "sdpa"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def load_fresh_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        attn_implementation=ATTN_IMPL,
    )
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(
        r=32, lora_alpha=64,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

model = load_fresh_model()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

trainable params: 83,886,080 || all params: 7,331,909,632 || trainable%: 1.1441


## 3. Dataset & Training

In [13]:
class TextDataset(Dataset):
    # Dataset with prompt-masked labels for causal LM training
    def __init__(self, texts, prompt_lens, tokenizer, max_length):
        self.items = []
        for text, plen in zip(texts, prompt_lens):
            enc = tokenizer(
                text, truncation=True, max_length=max_length,
                padding="max_length", return_tensors="pt",
            )
            input_ids = enc['input_ids'].squeeze()
            attn_mask = enc['attention_mask'].squeeze()
            labels = input_ids.clone()
            labels[:min(plen, max_length)] = -100  # mask prompt
            labels[attn_mask == 0] = -100  # mask padding
            self.items.append({
                'input_ids': input_ids,
                'attention_mask': attn_mask,
                'labels': labels,
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

In [14]:
BATCH_SIZE = 16
LR = 2e-4

def train_on_block(model, texts, prompt_lens, block_name, num_epochs):
    # Train model on one block. Returns (loss, elapsed).
    dataset = TextDataset(texts, prompt_lens, tokenizer, MAX_SEQ_LEN)
    print(f"  Training {len(dataset)} examples, {num_epochs} epochs...")

    args = TrainingArguments(
        output_dir=f"/tmp/ckpt_{block_name}",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=1,
        learning_rate=LR,
        bf16=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        optim="paged_adamw_8bit",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        seed=SEED,
        dataloader_pin_memory=True,
        dataloader_num_workers=4,
        gradient_checkpointing=False,
    )

    trainer = Trainer(
        model=model, train_dataset=dataset, args=args,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    start = time.time()
    result = trainer.train()
    elapsed = time.time() - start
    print(f"  {block_name}: loss={result.training_loss:.4f}, time={elapsed:.0f}s")
    return result.training_loss, elapsed

## 4. Evaluation Functions

In [15]:
@torch.no_grad()
def evaluate_loss(model, texts, prompt_lens, max_samples=100):
    # Compute average loss and perplexity on response tokens only
    model.eval()
    indices = random.sample(range(len(texts)), min(max_samples, len(texts)))
    total_loss, total_tokens = 0.0, 0

    for idx in indices:
        enc = tokenizer(
            texts[idx], truncation=True,
            max_length=MAX_SEQ_LEN, return_tensors="pt",
        ).to(model.device)
        labels = enc['input_ids'].clone()
        plen = min(prompt_lens[idx], labels.shape[1])
        labels[0, :plen] = -100
        labels[enc['attention_mask'] == 0] = -100

        outputs = model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            labels=labels,
        )
        n = (labels != -100).sum().item()
        if n > 0:
            total_loss += outputs.loss.item() * n
            total_tokens += n

    avg_loss = total_loss / total_tokens if total_tokens > 0 else float('inf')
    ppl = np.exp(min(avg_loss, 100))
    model.train()
    return avg_loss, ppl

In [16]:
@torch.no_grad()
def evaluate_generation(model, entries, max_samples=100):
    # Evaluate API-call generation accuracy
    # Returns (api_name_accuracy, full_accuracy)
    model.eval()
    if len(entries) > max_samples:
        entries = random.sample(entries, max_samples)

    system_prompt = config['system_prompt']
    name_correct, full_correct, total = 0, 0, 0

    for entry in entries:
        expected = entry.get('output', '')
        match = re.search(r'\[([A-Za-z_][A-Za-z0-9_]*)\(', expected)
        if not match:
            continue
        expected_api = match.group(1)
        expected_params = dict(re.findall(r"(\w+)='([^']*)'", expected))

        inp = entry['input']
        prompt = f"[INST] {system_prompt}\n\n{inp} [/INST]"

        enc = tokenizer(
            prompt, truncation=True,
            max_length=MAX_SEQ_LEN - 128, return_tensors="pt",
        ).to(model.device)

        gen = model.generate(
            **enc, max_new_tokens=128,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
        generated = tokenizer.decode(
            gen[0][enc['input_ids'].shape[1]:], skip_special_tokens=True
        )

        if expected_api.lower() in generated.lower():
            name_correct += 1
            if expected_params:
                gen_params = dict(re.findall(r"(\w+)='([^']*)'", generated))
                if any(
                    gen_params.get(k, '').lower() == v.lower()
                    for k, v in expected_params.items()
                ):
                    full_correct += 1
            else:
                full_correct += 1
        total += 1

    model.train()
    name_acc = name_correct / total if total > 0 else 0.0
    full_acc = full_correct / total if total > 0 else 0.0
    return name_acc, full_acc

## 5. Zero-Shot Baseline

In [17]:
print("=" * 60)
print("ZERO-SHOT BASELINE")
print("=" * 60)

zero_shot = {'loss': [], 'ppl': [], 'name_acc': [], 'full_acc': []}

for j in range(NUM_BLOCKS):
    eval_texts, eval_plens = get_eval_texts(blocks[j], CONDITION)
    loss, ppl = evaluate_loss(model, eval_texts, eval_plens)
    name_acc, full_acc = evaluate_generation(model, blocks[j]['eval_entries_raw'])
    zero_shot['loss'].append(loss)
    zero_shot['ppl'].append(ppl)
    zero_shot['name_acc'].append(name_acc)
    zero_shot['full_acc'].append(full_acc)
    print(f"  D{j+1}: loss={loss:.3f}, ppl={ppl:.1f}, "
          f"name={name_acc:.1%}, full={full_acc:.1%}")

ZERO-SHOT BASELINE
  D1: loss=1.468, ppl=4.3, name=90.5%, full=79.4%
  D2: loss=1.509, ppl=4.5, name=78.7%, full=65.6%
  D3: loss=1.611, ppl=5.0, name=85.0%, full=56.7%
  D4: loss=1.456, ppl=4.3, name=87.5%, full=68.8%
  D5: loss=1.367, ppl=3.9, name=87.5%, full=68.8%
  D6: loss=1.541, ppl=4.7, name=84.5%, full=66.2%


## 6. Continual Learning Loop

In [18]:
print(f"\n{'=' * 60}")
print(f"CONTINUAL LEARNING — Condition {CONDITION}, Seed {SEED}")
print(f"{'=' * 60}")

eval_loss_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
eval_ppl_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
eval_acc_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
eval_full_acc_mat = np.zeros((NUM_BLOCKS, NUM_BLOCKS))
train_losses, train_times, epochs_per_block = [], [], []

CHECKPOINT_DIR = f"checkpoints_{CONDITION}_seed{SEED}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
experiment_start = time.time()

for i in range(NUM_BLOCKS):
    print(f"\n--- Training on D{i+1}/{NUM_BLOCKS} ---")
    train_texts, train_plens = get_train_texts(blocks[i], CONDITION)
    epochs = get_epochs(blocks[i], CONDITION)
    epochs_per_block.append(epochs)

    t_loss, t_time = train_on_block(
        model, train_texts, train_plens,
        block_name=f"{CONDITION}_s{SEED}_D{i+1}",
        num_epochs=epochs,
    )
    train_losses.append(t_loss)
    train_times.append(t_time)

    print(f"  Evaluating all blocks...")
    for j in range(NUM_BLOCKS):
        eval_texts, eval_plens = get_eval_texts(blocks[j], CONDITION)
        loss, ppl = evaluate_loss(model, eval_texts, eval_plens)
        name_acc, full_acc = evaluate_generation(model, blocks[j]['eval_entries_raw'])
        eval_loss_mat[i][j] = loss
        eval_ppl_mat[i][j] = ppl
        eval_acc_mat[i][j] = name_acc
        eval_full_acc_mat[i][j] = full_acc
        tag = "(curr)" if j == i else "(prev)" if j < i else "(fut)"
        print(f"    D{j+1} {tag}: loss={loss:.3f}, ppl={ppl:.1f}, "
              f"name={name_acc:.1%}, full={full_acc:.1%}")

    # Checkpoint after each block
    ckpt = {
        'condition': CONDITION, 'seed': SEED,
        'blocks_trained': i + 1,
        'eval_loss': eval_loss_mat[:i+1].tolist(),
        'eval_ppl': eval_ppl_mat[:i+1].tolist(),
        'eval_acc': eval_acc_mat[:i+1].tolist(),
        'eval_full_acc': eval_full_acc_mat[:i+1].tolist(),
        'train_losses': train_losses, 'train_times': train_times,
    }
    with open(f'{CHECKPOINT_DIR}/after_D{i+1}.json', 'w') as f:
        json.dump(ckpt, f, indent=2)
    print(f"  Checkpoint saved")

total_time = time.time() - experiment_start
print(f"\nTotal: {total_time:.0f}s ({total_time/60:.1f} min)")


CONTINUAL LEARNING — Condition B, Seed 42

--- Training on D1/6 ---


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Training 448 examples, 3 epochs...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,0.606123
20,0.271430
30,0.200231
40,0.132753
50,0.118499
60,0.082098
70,0.063394
80,0.058046


...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Training 463 examples, 3 epochs...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,0.209514
20,0.151904
30,0.102008
40,0.055963
50,0.048871
60,0.042187
70,0.029668
80,0.030436


  B_s42_D6: loss=0.0795, time=1099s
  Evaluating all blocks...
    D1 (prev): loss=0.764, ppl=2.1, name=100.0%, full=96.8%
    D2 (prev): loss=0.813, ppl=2.3, name=100.0%, full=90.2%
    D3 (prev): loss=0.875, ppl=2.4, name=100.0%, full=93.3%
    D4 (prev): loss=0.774, ppl=2.2, name=98.4%, full=93.8%
    D5 (prev): loss=0.677, ppl=2.0, name=100.0%, full=95.3%
    D6 (curr): loss=0.801, ppl=2.2, name=100.0%, full=97.2%
  Checkpoint saved

Total: 16330s (272.2 min)


## 7. Save Final Results

In [19]:
results = {
    'condition': CONDITION,
    'seed': SEED,
    'zero_shot': zero_shot,
    'eval_loss': eval_loss_mat.tolist(),
    'eval_ppl': eval_ppl_mat.tolist(),
    'eval_acc': eval_acc_mat.tolist(),
    'eval_full_acc': eval_full_acc_mat.tolist(),
    'train_losses': train_losses,
    'train_times': train_times,
    'epochs_per_block': epochs_per_block,
    'total_time': total_time,
    'config': {
        'model': MODEL_NAME,
        'num_blocks': NUM_BLOCKS,
        'base_epochs': BASE_EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'max_seq_len': MAX_SEQ_LEN,
        'lora_r': 32, 'lora_alpha': 64,
        'attn': ATTN_IMPL,
        'precision': 'bf16',
    },
}

output_file = f"results_{CONDITION}_seed{SEED}.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved: {output_file}")


Results saved: results_B_seed42.json
